# Nemotron Reasoning Challenge — v5

**Wat is nieuw in v5:**

- **Werkende Triton-fix** ingebakken (PATH + env-vars + module unload, dit werkt op Blackwell GPU's)
- **6-template aanpak**: we hebben ontdekt dat de competitie-data 6 mechanische templates gebruikt,
  niet 4. Generators-stubs voor alle 6.
- **Data-exploratie cel** die per template 3 voorbeelden toont, zodat je de variabele/vaste delen
  ziet voordat je de generators invult.
- **Smoke-test mode** standaard aan om snel pipeline te valideren.

**Hoe te gebruiken:**

1. Run alles van boven naar beneden met `SMOKE_TEST = True`
2. Bij cel 9 (templates inspectie): **stop hier even** en bekijk de output goed. 
   Schrijf op wat het variabele en vaste deel per template is.
3. Pas de generators in cel 11 aan op basis van wat je in cel 9 zag.
4. Run de rest.
5. Als de smoke test slaagt: zet `SMOKE_TEST = False` en run alles opnieuw.


## 0. Triton ptxas-fix

Deze cel **moet als eerste** — vóór alle imports van transformers, torch, etc.
Hij kopieert de ptxas binaries naar een writable locatie, zet de PATH zodat Triton
ze daar vindt, en lost eventuele al-geladen Triton/mamba modules zodat ze opnieuw 
worden geïmporteerd met de juiste paden.

In [ ]:
import os
import sys
import shutil

# Stap 1: Kopieer triton binaries naar writable locatie
SRC_DIR = '/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/triton/backends/nvidia/bin'
DST_DIR = '/kaggle/working/triton_bin'

if os.path.exists(SRC_DIR):
    os.makedirs(DST_DIR, exist_ok=True)
    for f in os.listdir(SRC_DIR):
        src = os.path.join(SRC_DIR, f)
        dst = os.path.join(DST_DIR, f)
        if not os.path.exists(dst):
            shutil.copy2(src, dst)
        os.chmod(dst, 0o755)

# Stap 2: PATH voorop, plus expliciete env-vars
os.environ['PATH'] = '/kaggle/working/triton_bin:' + os.environ.get('PATH', '')
os.environ['TRITON_PTXAS_PATH'] = '/kaggle/working/triton_bin/ptxas-blackwell'
os.environ['TRITON_CUOBJDUMP_PATH'] = '/kaggle/working/triton_bin/cuobjdump'
os.environ['TRITON_NVDISASM_PATH'] = '/kaggle/working/triton_bin/nvdisasm'

# Stap 3: Triton cache wissen
for d in [os.path.expanduser('~/.triton'), '/tmp/triton', '/root/.triton']:
    if os.path.exists(d):
        shutil.rmtree(d, ignore_errors=True)

# Stap 4: Forceer unload van al geladen triton/mamba modules
for mod_name in list(sys.modules.keys()):
    if 'triton' in mod_name or 'mamba_ssm' in mod_name:
        del sys.modules[mod_name]

# Verifieer
print(f"PATH start: {os.environ['PATH'][:100]}")
print(f"PTXAS exists & exec: {os.access('/kaggle/working/triton_bin/ptxas-blackwell', os.X_OK)}")
print("Triton/mamba modules unloaded — worden opnieuw geladen bij volgende import")

# GPU-info
import torch
print(f"\nGPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


## 1. Configuratie

In [ ]:
# === MODE ===
SMOKE_TEST = True   # True = mini-run (~10 min); False = volledige run

# === DATA ===
if SMOKE_TEST:
    N_GENERATED_PER_TYPE = 30      # 6 types × 30 = 180 generated samples
    N_ORIGINAL_SAMPLES = 200
    NUM_EPOCHS = 1
else:
    N_GENERATED_PER_TYPE = 250     # 6 types × 250 = 1500 generated samples
    N_ORIGINAL_SAMPLES = 1200
    NUM_EPOCHS = 2

USE_GENERATED_DATA = True
USE_ORIGINAL_TRAIN = True

# === MODEL ===
MODEL_NAME = "metric/nemotron-3-nano-30b-a3b-bf16/transformers/default"

# === LORA — afgestemd op winnende submission-logboek ===
LORA_RANK = 32
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
TARGET_MODULES = "all-linear"

# === TRAINING ===
LEARNING_RATE = 5e-5
BATCH_SIZE = 1
GRAD_ACCUM = 4
MAX_SEQ_LEN = 2048

# === PATHS ===
DATA_PATH = "/kaggle/input/nvidia-nemotron-3-reasoning-challenge/train.csv"
OUTPUT_DIR = "/kaggle/working"
EXPERIMENT_LOG = f"{OUTPUT_DIR}/experiments.csv"

SEED = 42

print(f"Mode: {'SMOKE_TEST' if SMOKE_TEST else 'PRODUCTION'}")
print(f"Total samples target: ~{N_GENERATED_PER_TYPE * 6 + N_ORIGINAL_SAMPLES}")
print(f"Epochs: {NUM_EPOCHS}")


In [ ]:
import random
import re
import json
from datetime import datetime
from pathlib import Path

random.seed(SEED)
torch.manual_seed(SEED)


## 2. Data laden

In [ ]:
import polars as pl

train_raw = pl.read_csv(DATA_PATH)
print(f"Aantal samples in train.csv: {len(train_raw)}")
train_raw.head(3)


## 3. Template-detectie

We weten dat de data ~6 mechanische templates volgt. We groeperen elke prompt
op zijn eerste 8 woorden. De top 6 daarvan zijn onze templates.

In [ ]:
# Groepeer op eerste 8 woorden = "template fingerprint"
def template_key(p):
    return " ".join(p.split()[:8])

train_raw = train_raw.with_columns(
    pl.col("prompt").map_elements(template_key, return_dtype=pl.String).alias("template")
)

print("Top 10 templates:")
top_templates = train_raw.group_by("template").len().sort("len", descending=True).head(10)
print(top_templates)

# Pak de top 6 als onze officiële template-set
TEMPLATES = top_templates.head(6)["template"].to_list()
print(f"\n✓ {len(TEMPLATES)} templates geïdentificeerd")


## 4. Template inspectie — KIJK HIER GOED NAAR

Per template tonen we **3 voorbeelden + antwoord**. Schrijf voor jezelf op:
- Wat is het **vaste deel** van de prompt (woord-voor-woord hetzelfde)?
- Wat is het **variabele deel** (verschilt per sample)?
- Wat is de **regel** die antwoord uit prompt afleidt?

Dit is de meest waardevolle 5 minuten van je hele competitie-deelname. 
Zonder deze stap zijn de generators in cel 11 een gok.

In [ ]:
# Toon per template 3 voorbeelden
for i, tmpl in enumerate(TEMPLATES, 1):
    print("=" * 80)
    print(f"TEMPLATE {i}: {tmpl}...")
    print("=" * 80)
    samples = train_raw.filter(pl.col("template") == tmpl).sample(n=3, seed=SEED).to_dicts()
    for j, s in enumerate(samples, 1):
        print(f"\n--- Sample {j} ---")
        print(f"PROMPT: {s['prompt']}")
        print(f"ANSWER: {s['answer']}")
    print()


## 5. Generators — placeholders voor 6 templates

⚠️ **De generators hieronder zijn placeholders.** Op basis van wat je in cel 9 hebt gezien,
pas je elke generator aan zodat 'ie:
1. De prompt-stijl van dat template nabootst
2. Een geldig antwoord per de regel produceert
3. Een chain-of-thought genereert die naar dat antwoord redeneert

Voor elke generator: 
- `prompt` moet beginnen met exact dezelfde opening als het template
- `reasoning` moet eindigen op `\\boxed{answer}`
- `answer` moet matchen met de regel die je in cel 9 hebt herkend

**Tip:** schrijf eerst één generator écht goed, run de hele notebook, kijk naar val_accuracy 
voor dat task_type. Als die >70% is, generaliseer dezelfde aanpak naar de andere templates.

In [ ]:
# ===========================================================
# Placeholder-generators. Vervang elk met een echte versie 
# op basis van wat je in cel 9 hebt geanalyseerd.
# ===========================================================

def int_to_roman(n: int) -> str:
    vals = [(1000,'M'),(900,'CM'),(500,'D'),(400,'CD'),(100,'C'),(90,'XC'),
            (50,'L'),(40,'XL'),(10,'X'),(9,'IX'),(5,'V'),(4,'IV'),(1,'I')]
    out = ''
    for v, s in vals:
        while n >= v:
            out += s
            n -= v
    return out

# Template 1 — vervang inhoud op basis van cel 9 output
def gen_template_1():
    # Placeholder: binary encoding-puzzel
    chars = ['cat', 'dog', 'bird', 'fish', 'tree', 'star']
    mapping = {c: random.randint(0, 1) for c in chars}
    seq = [random.choice(chars) for _ in range(random.randint(6, 10))]
    answer = ''.join(str(mapping[c]) for c in seq)
    
    rules = ', '.join(f"'{c}' is {mapping[c]}" for c in chars)
    prompt = (
        f"In Alice's Wonderland, a secret code uses these mappings: {rules}. "
        f"Encode the sequence: {' -> '.join(seq)}. "
        "Provide the binary string."
    )
    reasoning = "Let me decode each element.\n"
    for c in seq:
        reasoning += f"  '{c}' -> {mapping[c]}\n"
    reasoning += f"Concatenated: {answer}\n\\boxed{{{answer}}}"
    return prompt, answer, reasoning


def gen_template_2():
    # Placeholder: gravity / number puzzle
    a = random.randint(10, 80)
    b = random.randint(5, 30)
    result = a + b
    answer = str(result)
    prompt = (
        f"In Alice's Wonderland, the gravity of numbers pulls {a} and {b} together. "
        f"Compute their sum."
    )
    reasoning = f"I add the numbers.\n{a} + {b} = {result}\n\\boxed{{{answer}}}"
    return prompt, answer, reasoning


def gen_template_3():
    # Placeholder: another binary variant
    return gen_template_1()  # voorlopig identiek aan template 1


def gen_template_4():
    # Placeholder: short phrase (SVO)
    subjects = ['cat', 'dog', 'wizard', 'queen']
    verbs = ['imagines', 'creates', 'sees', 'dreams']
    objects = ['book', 'secret', 'door', 'mirror']
    s, v, o = random.choice(subjects), random.choice(verbs), random.choice(objects)
    answer = f"{s} {v} {o}"
    prompt = (
        f"In Alice's Wonderland, secret events follow simple rules. "
        f"A {s} performs the action '{v}' on a {o}. Describe in three words."
    )
    reasoning = (
        f"Subject: {s}\nVerb: {v}\nObject: {o}\n"
        f"Combined: {answer}\n\\boxed{{{answer}}}"
    )
    return prompt, answer, reasoning


def gen_template_5():
    # Placeholder: roman numeral
    a = random.randint(10, 80)
    b = random.randint(5, 30)
    result = a + b
    answer = int_to_roman(result)
    prompt = (
        f"In Alice's Wonderland, numbers are spoken as Roman numerals. "
        f"Compute the sum of {a} and {b} as a Roman numeral."
    )
    reasoning = (
        f"I compute {a} + {b} = {result}.\n"
        f"In Roman numerals: {answer}\n"
        f"\\boxed{{{answer}}}"
    )
    return prompt, answer, reasoning


def gen_template_6():
    # Placeholder: variant van template 1
    return gen_template_1()  # voorlopig identiek


GENERATORS = {
    'template_1': gen_template_1,
    'template_2': gen_template_2,
    'template_3': gen_template_3,
    'template_4': gen_template_4,
    'template_5': gen_template_5,
    'template_6': gen_template_6,
}

# Test alle generators
for name, gen in GENERATORS.items():
    p, a, r = gen()
    print(f"=== {name} ===")
    print(f"PROMPT: {p[:120]}...")
    print(f"ANSWER: {a}")
    print()


## 6. Dataset opbouwen

In [ ]:
def build_generated_dataset(n_per_type):
    rows = []
    for name, gen in GENERATORS.items():
        for _ in range(n_per_type):
            prompt, answer, reasoning = gen()
            rows.append({
                'prompt': prompt, 'answer': answer, 'reasoning': reasoning,
                'task_type': name, 'source': 'generated',
            })
    return rows

def build_original_dataset(n_samples=None):
    df = train_raw
    if n_samples is not None and n_samples < len(df):
        df = df.sample(n=n_samples, seed=SEED)
    rows = []
    for r in df.to_dicts():
        reasoning = (
            f"Let me work through this puzzle step by step.\n"
            f"I extract the rule from the prompt and apply it carefully.\n"
            f"This leads to: {r['answer']}\n"
            f"\\boxed{{{r['answer']}}}"
        )
        rows.append({
            'prompt': r['prompt'], 'answer': r['answer'], 'reasoning': reasoning,
            'task_type': r.get('template', 'original'), 'source': 'original',
        })
    return rows

all_rows = []
if USE_GENERATED_DATA:
    gen_rows = build_generated_dataset(N_GENERATED_PER_TYPE)
    all_rows.extend(gen_rows)
    print(f"Gegenereerde samples: {len(gen_rows)}")

if USE_ORIGINAL_TRAIN:
    orig_rows = build_original_dataset(N_ORIGINAL_SAMPLES)
    all_rows.extend(orig_rows)
    print(f"Originele samples: {len(orig_rows)}")

random.shuffle(all_rows)
print(f"\nTotaal: {len(all_rows)} samples")


## 7. Model + LoRA

In [ ]:
import site
cutlass_pkg_path = "/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/nvidia_cutlass_dsl/python_packages/"
if os.path.exists(cutlass_pkg_path):
    site.addsitedir(cutlass_pkg_path)

import kagglehub
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model, TaskType

MODEL_PATH = kagglehub.model_download(MODEL_NAME)
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    device_map="auto",
    trust_remote_code=True,
    dtype=torch.bfloat16,
)
print("Base model geladen.")

lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    target_modules=TARGET_MODULES,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


## 8. Format voor chat template

In [ ]:
SYSTEM_PROMPT = (
    "You are a reasoning assistant. Solve the puzzle step by step. "
    "Always end your response with the final answer in \\boxed{} format."
)

def format_sample(row):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": row["prompt"]},
        {"role": "assistant", "content": row["reasoning"]},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False)

formatted = [format_sample(r) for r in all_rows]

val_size = max(50, len(formatted) // 20)
val_texts = formatted[:val_size]
train_texts = formatted[val_size:]
val_rows = all_rows[:val_size]

from datasets import Dataset
train_ds = Dataset.from_dict({"text": train_texts})
val_ds = Dataset.from_dict({"text": val_texts})

print(f"Train: {len(train_ds)}, Val: {len(val_ds)}")
print(f"\nVoorbeeld:\n{formatted[0][:800]}")


## 9. Training

`gradient_checkpointing=False` om Mamba Triton-compileer-issues te vermijden.
Op een 96GB GPU past dit prima.

In [ ]:
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling
import time

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=MAX_SEQ_LEN,
        padding=False,
    )

train_tokenized = train_ds.map(tokenize_function, batched=True, remove_columns=["text"])
val_tokenized = val_ds.map(tokenize_function, batched=True, remove_columns=["text"])
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

training_args = TrainingArguments(
    output_dir=f"{OUTPUT_DIR}/checkpoints",
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    bf16=True,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="no",
    report_to="none",
    gradient_checkpointing=False,    # was True; gaf Mamba Triton-fout
    optim="adamw_torch",
    seed=SEED,
    remove_unused_columns=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    data_collator=data_collator,
)

t0 = time.time()
trainer.train()
train_time_min = (time.time() - t0) / 60
print(f"\nTraining klaar in {train_time_min:.1f} min")

final_eval_loss = None
for log in reversed(trainer.state.log_history):
    if 'eval_loss' in log:
        final_eval_loss = log['eval_loss']
        break
print(f"Laatste eval loss: {final_eval_loss}")


## 10. Accuracy meten

In [ ]:
def extract_boxed(text):
    matches = re.findall(r"\\boxed\{([^{}]*)\}", text)
    if matches:
        return matches[-1].strip()
    return text.strip().split('\n')[-1].strip()

def evaluate_accuracy(model, val_rows, n_samples=30):
    model.eval()
    samples = random.sample(val_rows, min(n_samples, len(val_rows)))
    correct = 0
    examples = []
    for row in samples:
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": row["prompt"]},
        ]
        inputs = tokenizer.apply_chat_template(
            messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
        ).to(model.device)
        with torch.no_grad():
            output = model.generate(
                inputs, max_new_tokens=512, do_sample=False,
                temperature=0.0, pad_token_id=tokenizer.eos_token_id,
            )
        response = tokenizer.decode(output[0][inputs.shape[1]:], skip_special_tokens=True)
        predicted = extract_boxed(response)
        is_correct = predicted.strip() == row["answer"].strip()
        if is_correct:
            correct += 1
        examples.append({
            'task_type': row['task_type'],
            'expected': row['answer'],
            'predicted': predicted,
            'correct': is_correct,
        })
    return correct / len(samples), examples

n_eval = 15 if SMOKE_TEST else 30
val_acc, val_examples = evaluate_accuracy(model, val_rows, n_samples=n_eval)
print(f"\nVal accuracy: {val_acc:.3f} ({int(val_acc * n_eval)}/{n_eval})")

print("\nPer task_type:")
by_type = {}
for ex in val_examples:
    by_type.setdefault(ex['task_type'], []).append(ex['correct'])
for tt, results in by_type.items():
    acc = sum(results) / len(results)
    print(f"  {tt}: {acc:.2%} ({sum(results)}/{len(results)})")

print("\nFouten (eerste 5):")
shown = 0
for ex in val_examples:
    if not ex['correct']:
        print(f"  [{ex['task_type']}] expected={ex['expected']!r}, got={ex['predicted']!r}")
        shown += 1
        if shown >= 5:
            break


## 11. Experiment log

In [ ]:
import csv

log_entry = {
    'timestamp': datetime.now().isoformat(timespec='seconds'),
    'mode': 'SMOKE' if SMOKE_TEST else 'PROD',
    'lr': LEARNING_RATE,
    'epochs': NUM_EPOCHS,
    'lora_rank': LORA_RANK,
    'target_modules': TARGET_MODULES,
    'n_generated_per_type': N_GENERATED_PER_TYPE if USE_GENERATED_DATA else 0,
    'n_original': N_ORIGINAL_SAMPLES if USE_ORIGINAL_TRAIN else 0,
    'total_samples': len(all_rows),
    'eval_loss': round(final_eval_loss, 4) if final_eval_loss else None,
    'val_accuracy': round(val_acc, 4),
    'train_time_min': round(train_time_min, 1),
}

write_header = not os.path.exists(EXPERIMENT_LOG)
with open(EXPERIMENT_LOG, 'a', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=log_entry.keys())
    if write_header:
        writer.writeheader()
    writer.writerow(log_entry)

print("Deze run:")
for k, v in log_entry.items():
    print(f"  {k}: {v}")

print(f"\n=== ALLE EXPERIMENTEN ===")
with open(EXPERIMENT_LOG) as f:
    print(f.read())


## 12. Submission

In [ ]:
if SMOKE_TEST:
    print("SMOKE_TEST mode — submission overgeslagen.")
    print("Zet SMOKE_TEST=False bovenaan en run alles opnieuw voor echte submission.")
else:
    import subprocess
    ckpt_dir = f"{OUTPUT_DIR}/checkpoints"
    if os.path.exists(ckpt_dir):
        shutil.rmtree(ckpt_dir)

    print(f"Saving adapter to {OUTPUT_DIR}...")
    model.save_pretrained(OUTPUT_DIR)

    adapter_files = ["adapter_config.json", "adapter_model.safetensors"]
    existing = [f for f in adapter_files if os.path.exists(f"{OUTPUT_DIR}/{f}")]
    print(f"In submission.zip: {existing}")

    subprocess.run(
        f"cd {OUTPUT_DIR} && zip -m submission.zip " + " ".join(existing),
        shell=True, check=True
    )
    print(f"\n✓ submission.zip aangemaakt")
    print(f"Val accuracy: {val_acc:.3f}")


## Volgende stappen

1. **Smoke test geslaagd?** Bekijk val_accuracy per task_type. Zelfs als 't laag is, 
   weet je dat de pipeline werkt.
2. **Verbeter de generators** op basis van wat je in cel 9 hebt gezien. Eén template
   tegelijk perfect maken is beter dan alle 6 half doen.
3. **Bij elke iteratie**: noteer de val_accuracy in het experiment-log. Bouw je eigen 
   empirische tabel op.
4. **Productie-run**: `SMOKE_TEST = False`, run alles opnieuw.
